# Дообучение v2: Qwen2.5-Coder-7B на своём датасете «объяснение кода» (RU)

**Перед запуском** (Settings справа):
- Accelerator: **GPU T4 x1**, Internet: **On**
- **Add Input → Upload** → файл `explain_ru_train.jsonl` (из папки `Developer/coder`)

Что внутри:
- свой датасет (~1200 проверенных примеров: механизм, «почему», подвох, исправление) + 600 примеров glaive, чтобы модель не разучилась обычным ответам;
- половина примеров обучается **с системным промптом**, половина без — модель будет хорошо отвечать в обоих режимах;
- в конце — **честное сравнение на 18 отложенных вопросах**, которых нет в обучении, и экспорт GGUF.

Запуск: **Save Version → Save & Run All** (≈1,5–2 часа). Результаты: `eval_results.md`, `my-coder-v2-Q4_K_M.gguf`, `lora_adapter_v2` во вкладке Output.

In [ ]:
# 1. Установка
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install -q unsloth

In [ ]:
# 2. Настройки
MODEL_NAME  = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
N_GLAIVE    = 300     # общие примеры для регуляризации
EPOCHS      = 2
LR          = 1e-4
LORA_R      = 32
MAX_SEQ_LEN = 3072    # system-примеры длинные
SYSTEM_SHARE = 0.5    # доля примеров с системным промптом
OUT_NAME    = "my-coder-v2"
SEED        = 42

SYSTEM_PROMPT = """Ты — опытный преподаватель программирования. Когда тебя просят объяснить код:
1. Одной-двумя фразами скажи, что код делает в целом.
2. Назови механизм языка, на котором держится его поведение (например: замыкание и позднее связывание, изменяемый объект и общие ссылки, вычисление значения по умолчанию при определении функции, короткое вычисление and/or, ленивость генераторов), и объясни, ПОЧЕМУ код работает именно так — шаг за шагом, как это выполняет интерпретатор.
3. Если в коде есть подвох или неочевидное поведение — назови его прямо и покажи, когда он проявится. Если подвоха нет — так и скажи, не выдумывай.
4. Укажи важные ограничения и граничные случаи: типы аргументов, хешируемость, порядок элементов, сложность.
5. Если есть ошибка или плохая практика — покажи исправленный вариант кода.
Не пересказывай код построчно, если это не помогает понять суть. Пиши по-русски, точно и без воды."""

# Строка режима «ворчун» — ровно как в датасете (ds/tools/persona_data.json) и в loca (/grumpy)
PERSONA = "Режим «ворчун» включён: ты уставший циничный сеньор, который в сотый раз видит одни и те же косяки. Начинай ответ с короткой реакции (1–2 фразы) на конкретный косяк собеседника — можно с матом и подколками про «кожаного», но только про код. Дальше — полное точное объяснение, как обычно."


In [ ]:
# 3. Данные
import glob, json, random
from datasets import load_dataset, Dataset, concatenate_datasets

random.seed(SEED)
files = glob.glob("/kaggle/input/**/explain_ru*.jsonl", recursive=True)
assert files, "Загрузите explain_ru_train.jsonl через Add Input → Upload"
own = []
with open(files[0], encoding="utf-8") as f:
    for line in f:
        if line.strip():
            own.append(json.loads(line)["messages"])
print("Свой датасет:", len(own), "из", files[0])

def with_system(msgs):
    if msgs[0]["role"] == "system":
        # режим «ворчун»: строка режима есть всегда, обычный системный промпт — как у остальных
        extra = SYSTEM_PROMPT + "\n\n" if random.random() < SYSTEM_SHARE else ""
        return [{"role": "system", "content": extra + msgs[0]["content"]}] + msgs[1:]
    if random.random() < SYSTEM_SHARE:
        return [{"role": "system", "content": SYSTEM_PROMPT}] + msgs
    return msgs

own_rows = [{"messages": with_system(m)} for m in own]
print("из них в режиме «ворчун»:", sum(m[0]["role"] == "system" for m in own))

g = load_dataset("glaiveai/glaive-code-assistant-v2", split="train").shuffle(seed=SEED)
g = g.filter(lambda x: x["question"] and x["answer"] and len(x["answer"]) < 4000).select(range(N_GLAIVE))
g_rows = [{"messages": [{"role": "user", "content": x["question"]},
                        {"role": "assistant", "content": x["answer"]}]} for x in g]

ds = Dataset.from_list(own_rows + g_rows).shuffle(seed=SEED)
print(ds)
print(ds[0]["messages"][-1]["content"][:800])

In [ ]:
# 4. Модель + LoRA
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(MODEL_NAME, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)

In [ ]:
# 5. Шаблон чата
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
ds = ds.map(lambda b: {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in b["messages"]]},
            batched=True, remove_columns=ds.column_names)
lens = sorted(len(tokenizer(t)["input_ids"]) for t in ds["text"][:400])
print("токенов в примере: медиана", lens[len(lens)//2], "· 95%", lens[int(len(lens)*0.95)], "· макс", lens[-1])

In [ ]:
# 6. Обучение
import dataclasses
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

fields = {f.name for f in dataclasses.fields(SFTConfig)}
len_key = "max_length" if "max_length" in fields else "max_seq_length"
args = SFTConfig(**{len_key: MAX_SEQ_LEN}, dataset_text_field="text",
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=EPOCHS, learning_rate=LR, warmup_steps=20,
    lr_scheduler_type="cosine", weight_decay=0.01,
    fp16=True, bf16=False, optim="adamw_8bit",
    logging_steps=10, save_strategy="no", output_dir="/kaggle/working/out",
    report_to="none", seed=SEED)
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds, args=args)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=args)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
print(trainer.train())

In [ ]:
# 7. Сохранить адаптер сразу (страховка)
model.save_pretrained("/kaggle/working/lora_adapter_v2")
tokenizer.save_pretrained("/kaggle/working/lora_adapter_v2")

In [ ]:
# 8. Честная проверка на 18 отложенных вопросах (их нет в обучении)
FastLanguageModel.for_inference(model)
EVAL = [
 {
  "id": "e01",
  "q": "Объясни, что делает этот код и почему он работает:\n\n```python\ndef f(xs):\n    seen = set()\n    return [x for x in xs if not (x in seen or seen.add(x))]\n```"
 },
 {
  "id": "e02",
  "q": "Почему этот код ведёт себя неожиданно при повторных вызовах?\n\n```python\ndef add(item, items=[]):\n    items.append(item)\n    return items\n```"
 },
 {
  "id": "e03",
  "q": "Объясни, что выведет код и почему:\n\n```python\nfuncs = [lambda: i for i in range(3)]\nprint([f() for f in funcs])\n```"
 },
 {
  "id": "e04",
  "q": "Объясни простыми словами, как работает этот декоратор:\n\n```python\nimport functools\ndef cache(fn):\n    memo = {}\n    @functools.wraps(fn)\n    def wrapper(*args):\n        if args not in memo:\n            memo[args] = fn(*args)\n        return memo[args]\n    return wrapper\n```"
 },
 {
  "id": "e05",
  "q": "Что выведет этот код? Объясни.\n\n```python\ngrid = [[0] * 3] * 3\ngrid[0][0] = 1\nprint(grid)\n```"
 },
 {
  "id": "e06",
  "q": "Почему тут ошибка, ведь x объявлен выше?\n\n```python\nx = 10\ndef show():\n    print(x)\n    x = 5\nshow()\n```"
 },
 {
  "id": "e07",
  "q": "Почему второй print выводит 0?\n\n```python\nsquares = (n * n for n in range(4))\nprint(sum(squares))\nprint(sum(squares))\n```"
 },
 {
  "id": "e08",
  "q": "Почему результат False? Как правильно сравнивать?\n\n```python\nprint(0.1 + 0.2 == 0.3)\n```"
 },
 {
  "id": "e09",
  "q": "Что выведет этот код и почему? Как исправить?\n\n```javascript\nfor (var i = 0; i < 3; i++) {\n  setTimeout(() => console.log(i), 0);\n}\n```"
 },
 {
  "id": "e10",
  "q": "Почему этот запрос ничего не возвращает, хотя клиенты без заказов есть?\n\n```sql\nCREATE TABLE clients(id INTEGER, name TEXT);\nCREATE TABLE orders(id INTEGER, client_id INTEGER);\nINSERT INTO clients VALUES (1,'Аня'),(2,'Боря'),(3,'Вика');\nINSERT INTO orders VALUES (10,1),(11,NULL);\nSELECT name FROM clients WHERE id NOT IN (SELECT client_id FROM orders);\n```"
 },
 {
  "id": "e11",
  "q": "Объясни, как работает эта функция и какая у неё сложность:\n\n```python\ndef find(arr, target):\n    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if arr[mid] == target:\n            return mid\n        if arr[mid] < target:\n            lo = mid + 1\n        else:\n            hi = mid - 1\n    return -1\n```"
 },
 {
  "id": "e12",
  "q": "Объясни, как устроена и как работает эта программа целиком:\n\n```python\nimport json\n\nclass Library:\n    def __init__(self, path):\n        self.path = path\n        self.books = self._load()\n\n    def _load(self):\n        try:\n            with open(self.path, encoding='utf-8') as f:\n                return json.load(f)\n        except FileNotFoundError:\n            return {}\n\n    def _save(self):\n        with open(self.path, 'w', encoding='utf-8') as f:\n            json.dump(self.books, f, ensure_ascii=False, indent=2)\n\n    def add(self, isbn, title):\n        self.books[isbn] = {'title': title, 'taken_by': None}\n        self._save()\n\n    def take(self, isbn, reader):\n        book = self.books.get(isbn)\n        if book is None or book['taken_by']:\n            return False\n        book['taken_by'] = reader\n        self._save()\n        return True\n\n    def give_back(self, isbn):\n        self.books[isbn]['taken_by'] = None\n        self._save()\n\nlib = Library('books.json')\nlib.add('111', 'Мастер и Маргарита')\nprint(lib.take('111', 'Петя'), lib.take('111', 'Маша'))\n\n```"
 },
 {
  "id": "e13",
  "q": "Почему выручка после merge получилась больше реальной? Объясни и исправь.\n\n```python\nimport pandas as pd\norders = pd.DataFrame({'order_id': [1, 2, 3], 'client': ['A', 'B', 'A'], 'amount': [100, 250, 80]})\ncontacts = pd.DataFrame({'client': ['A', 'A', 'B'], 'phone': ['111', '222', '333']})\ndf = orders.merge(contacts, on='client')\nprint(df['amount'].sum())\n```"
 },
 {
  "id": "e14",
  "q": "Сделай дашборд на Streamlit для продаж интернет-магазина. Данные в sales.csv: date, region, category, revenue, orders. Нужны фильтры по региону и периоду, KPI (выручка, заказы, средний чек), график выручки по месяцам и топ-5 категорий. Объясни, как он устроен."
 },
 {
  "id": "e15",
  "q": "Напиши SQL-запрос: выручка по месяцам и рост в процентах к прошлому месяцу. Таблица orders(id, user_id, created_at TEXT 'YYYY-MM-DD', amount). Объясни, как он работает."
 },
 {
  "id": "e16",
  "q": "Почему модель почти не учится, хотя loss считается без ошибок?\n\n```python\nimport torch\nimport torch.nn as nn\n\ntorch.manual_seed(0)\nmodel = nn.Sequential(nn.Linear(4, 3), nn.Softmax(dim=1))\nloss_fn = nn.CrossEntropyLoss()\nx = torch.randn(8, 4)\ny = torch.randint(0, 3, (8,))\nopt = torch.optim.SGD(model.parameters(), lr=0.5)\nfor step in range(3):\n    loss = loss_fn(model(x), y)\n    loss.backward()\n    opt.step()\n    print(round(loss.item(), 4))\n```"
 },
 {
  "id": "e17",
  "q": "Объясни простыми словами, что делает model.eval() и torch.no_grad() и почему на валидации нужны оба. Что будет, если забыть одно из них?"
 },
 {
  "id": "e18",
  "q": "Я дообучил модель LoRA и смёрджил адаптер. Как понять, какие слои изменились сильнее всего? Напиши код, который сравнивает два state_dict (до и после) и выводит слои по убыванию изменения, и объясни, какие метрики тут осмысленны."
 }
]

def gen(question, system=None, max_new_tokens=700):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": question}]
    enc = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    ids = enc["input_ids"].to("cuda")
    out = model.generate(input_ids=ids, attention_mask=enc["attention_mask"].to("cuda"),
                         max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

results = []
for e in EVAL:
    with model.disable_adapter():
        base_p = gen(e["q"], SYSTEM_PROMPT)
    tuned = gen(e["q"])
    tuned_p = gen(e["q"], SYSTEM_PROMPT)
    results.append({"id": e["id"], "question": e["q"], "base+prompt": base_p, "tuned": tuned, "tuned+prompt": tuned_p})
    print(f"✓ {e['id']}")

with open("/kaggle/working/eval_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=1)
with open("/kaggle/working/eval_results.md", "w", encoding="utf-8") as f:
    for r in results:
        f.write(f"# {r['id']}\n\n{r['question']}\n\n")
        for k in ("base+prompt", "tuned", "tuned+prompt"):
            f.write(f"## {k}\n\n{r[k]}\n\n")
print("Готово: /kaggle/working/eval_results.md — пришлите этот файл")

In [ ]:
# 8b. Режим «ворчун»: включается строкой режима и НЕ протекает в обычные ответы
import re
MAT = re.compile(r"\bбля|пизд|\bху[йеё]|\bёб|\bеба|нахрен|кожан", re.I)
leak = [r["id"] for r in results if MAT.search(r["tuned"]) or MAT.search(r["tuned+prompt"])]
print("мат в обычном режиме (должно быть пусто):", leak or "нет ✓")

with open("/kaggle/working/eval_grumpy.md", "w", encoding="utf-8") as f:
    for e in EVAL[:6]:
        a = gen(e["q"], PERSONA)
        b = gen(e["q"], SYSTEM_PROMPT + "\n\n" + PERSONA)
        f.write(f"# {e['id']}\n\n{e['q']}\n\n## ворчун\n\n{a}\n\n## ворчун + промпт\n\n{b}\n\n")
        print(f"✓ {e['id']}:", a.split("\n")[0][:150])
print("Готово: /kaggle/working/eval_grumpy.md")


In [ ]:
# 9. Экспорт GGUF Q4_K_M
import shutil
os.chdir("/tmp")
model.save_pretrained_gguf(OUT_NAME, tokenizer, quantization_method="q4_k_m")
os.chdir("/kaggle/working")
cands = [p for p in glob.glob("/tmp/**/*.gguf", recursive=True) if "q4_k_m" in p.lower()]
print("Найдено:", cands)
if cands:
    shutil.copy(cands[0], f"/kaggle/working/{OUT_NAME}-Q4_K_M.gguf")
!ls -lh /kaggle/working